# 15 — Window Pattern

Purpose: use Spark window functions when row context matters.

Process schema:

```text
EVENTS
  |
  v
WINDOW PARTITION
  |
  +--> latest row per key
  +--> ranking
  +--> previous / next row
  +--> running totals
```

Typical use-cases:

```text
dedup latest record
rank items inside groups
compare current row with previous row
calculate running totals
```

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("window-pattern")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.warehouse.dir", "/tmp/spark_patterns_warehouse")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.version

'4.0.2'

## Step 1 — Input events

The dataset has:
- repeated users
- repeated event ids
- multiple events over time

In [2]:
schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("user_id", StringType(), False),
    StructField("event_type", StringType(), False),
    StructField("amount", DoubleType(), False),
    StructField("event_time_raw", StringType(), False),
    StructField("ingestion_time_raw", StringType(), False),
])

rows = [
    ("e1", "u1", "purchase", 10.0, "2026-01-01 10:00:00", "2026-01-01 10:01:00"),
    ("e2", "u1", "purchase", 15.0, "2026-01-01 10:05:00", "2026-01-01 10:06:00"),
    ("e2", "u1", "purchase", 17.0, "2026-01-01 10:05:00", "2026-01-01 10:07:00"),
    ("e3", "u1", "refund", 5.0, "2026-01-01 10:10:00", "2026-01-01 10:11:00"),
    ("e4", "u2", "purchase", 20.0, "2026-01-01 11:00:00", "2026-01-01 11:01:00"),
    ("e5", "u2", "purchase", 30.0, "2026-01-01 11:05:00", "2026-01-01 11:06:00"),
    ("e6", "u2", "refund", 8.0, "2026-01-01 11:10:00", "2026-01-01 11:11:00"),
]

events = (
    spark.createDataFrame(rows, schema)
    .withColumn("event_time", F.to_timestamp("event_time_raw"))
    .withColumn("ingestion_time", F.to_timestamp("ingestion_time_raw"))
    .drop("event_time_raw", "ingestion_time_raw")
)

events.orderBy("user_id", "event_time", "ingestion_time").show(truncate=False)

[Stage 0:>                                                          (0 + 2) / 2]

+--------+-------+----------+------+-------------------+-------------------+
|event_id|user_id|event_type|amount|event_time         |ingestion_time     |
+--------+-------+----------+------+-------------------+-------------------+
|e1      |u1     |purchase  |10.0  |2026-01-01 10:00:00|2026-01-01 10:01:00|
|e2      |u1     |purchase  |15.0  |2026-01-01 10:05:00|2026-01-01 10:06:00|
|e2      |u1     |purchase  |17.0  |2026-01-01 10:05:00|2026-01-01 10:07:00|
|e3      |u1     |refund    |5.0   |2026-01-01 10:10:00|2026-01-01 10:11:00|
|e4      |u2     |purchase  |20.0  |2026-01-01 11:00:00|2026-01-01 11:01:00|
|e5      |u2     |purchase  |30.0  |2026-01-01 11:05:00|2026-01-01 11:06:00|
|e6      |u2     |refund    |8.0   |2026-01-01 11:10:00|2026-01-01 11:11:00|
+--------+-------+----------+------+-------------------+-------------------+



## Step 2 — Latest record per key

Pattern:

```text
PARTITION BY event_id
ORDER BY ingestion_time DESC
rn = 1
```

This is useful for deduplication where the newest arrival wins.

In [3]:
latest_by_event_window = (
    Window
    .partitionBy("event_id")
    .orderBy(F.col("ingestion_time").desc())
)

ranked_events = events.withColumn("rn", F.row_number().over(latest_by_event_window))

latest_events = ranked_events.filter(F.col("rn") == 1).drop("rn")

ranked_events.orderBy("event_id", "rn").show(truncate=False)
latest_events.orderBy("event_id").show(truncate=False)

+--------+-------+----------+------+-------------------+-------------------+---+
|event_id|user_id|event_type|amount|event_time         |ingestion_time     |rn |
+--------+-------+----------+------+-------------------+-------------------+---+
|e1      |u1     |purchase  |10.0  |2026-01-01 10:00:00|2026-01-01 10:01:00|1  |
|e2      |u1     |purchase  |17.0  |2026-01-01 10:05:00|2026-01-01 10:07:00|1  |
|e2      |u1     |purchase  |15.0  |2026-01-01 10:05:00|2026-01-01 10:06:00|2  |
|e3      |u1     |refund    |5.0   |2026-01-01 10:10:00|2026-01-01 10:11:00|1  |
|e4      |u2     |purchase  |20.0  |2026-01-01 11:00:00|2026-01-01 11:01:00|1  |
|e5      |u2     |purchase  |30.0  |2026-01-01 11:05:00|2026-01-01 11:06:00|1  |
|e6      |u2     |refund    |8.0   |2026-01-01 11:10:00|2026-01-01 11:11:00|1  |
+--------+-------+----------+------+-------------------+-------------------+---+

+--------+-------+----------+------+-------------------+-------------------+
|event_id|user_id|event_type|am

## Step 3 — Rank events inside each user

Pattern:

```text
PARTITION BY user_id
ORDER BY amount DESC
```

This answers: highest value events per user.

In [4]:
amount_rank_window = (
    Window
    .partitionBy("user_id")
    .orderBy(F.col("amount").desc())
)

ranked_by_amount = (
    latest_events
    .withColumn("amount_rank", F.dense_rank().over(amount_rank_window))
)

ranked_by_amount.orderBy("user_id", "amount_rank").show(truncate=False)

+--------+-------+----------+------+-------------------+-------------------+-----------+
|event_id|user_id|event_type|amount|event_time         |ingestion_time     |amount_rank|
+--------+-------+----------+------+-------------------+-------------------+-----------+
|e2      |u1     |purchase  |17.0  |2026-01-01 10:05:00|2026-01-01 10:07:00|1          |
|e1      |u1     |purchase  |10.0  |2026-01-01 10:00:00|2026-01-01 10:01:00|2          |
|e3      |u1     |refund    |5.0   |2026-01-01 10:10:00|2026-01-01 10:11:00|3          |
|e5      |u2     |purchase  |30.0  |2026-01-01 11:05:00|2026-01-01 11:06:00|1          |
|e4      |u2     |purchase  |20.0  |2026-01-01 11:00:00|2026-01-01 11:01:00|2          |
|e6      |u2     |refund    |8.0   |2026-01-01 11:10:00|2026-01-01 11:11:00|3          |
+--------+-------+----------+------+-------------------+-------------------+-----------+



## Step 4 — Previous event per user

Pattern:

```text
lag(column) over user timeline
```

This answers: what happened before this event?

In [5]:
timeline_window = (
    Window
    .partitionBy("user_id")
    .orderBy("event_time")
)

with_previous = (
    latest_events
    .withColumn("previous_event_type", F.lag("event_type").over(timeline_window))
    .withColumn("previous_event_time", F.lag("event_time").over(timeline_window))
)

with_previous.orderBy("user_id", "event_time").show(truncate=False)

+--------+-------+----------+------+-------------------+-------------------+-------------------+-------------------+
|event_id|user_id|event_type|amount|event_time         |ingestion_time     |previous_event_type|previous_event_time|
+--------+-------+----------+------+-------------------+-------------------+-------------------+-------------------+
|e1      |u1     |purchase  |10.0  |2026-01-01 10:00:00|2026-01-01 10:01:00|NULL               |NULL               |
|e2      |u1     |purchase  |17.0  |2026-01-01 10:05:00|2026-01-01 10:07:00|purchase           |2026-01-01 10:00:00|
|e3      |u1     |refund    |5.0   |2026-01-01 10:10:00|2026-01-01 10:11:00|purchase           |2026-01-01 10:05:00|
|e4      |u2     |purchase  |20.0  |2026-01-01 11:00:00|2026-01-01 11:01:00|NULL               |NULL               |
|e5      |u2     |purchase  |30.0  |2026-01-01 11:05:00|2026-01-01 11:06:00|purchase           |2026-01-01 11:00:00|
|e6      |u2     |refund    |8.0   |2026-01-01 11:10:00|2026-01-

## Step 5 — Time since previous event

Use `lag` result to calculate gaps.

In [6]:
with_gap = (
    with_previous
    .withColumn(
        "minutes_since_previous_event",
        (F.col("event_time").cast("long") - F.col("previous_event_time").cast("long")) / 60.0
    )
)

with_gap.orderBy("user_id", "event_time").show(truncate=False)

+--------+-------+----------+------+-------------------+-------------------+-------------------+-------------------+----------------------------+
|event_id|user_id|event_type|amount|event_time         |ingestion_time     |previous_event_type|previous_event_time|minutes_since_previous_event|
+--------+-------+----------+------+-------------------+-------------------+-------------------+-------------------+----------------------------+
|e1      |u1     |purchase  |10.0  |2026-01-01 10:00:00|2026-01-01 10:01:00|NULL               |NULL               |NULL                        |
|e2      |u1     |purchase  |17.0  |2026-01-01 10:05:00|2026-01-01 10:07:00|purchase           |2026-01-01 10:00:00|5.0                         |
|e3      |u1     |refund    |5.0   |2026-01-01 10:10:00|2026-01-01 10:11:00|purchase           |2026-01-01 10:05:00|5.0                         |
|e4      |u2     |purchase  |20.0  |2026-01-01 11:00:00|2026-01-01 11:01:00|NULL               |NULL               |NULL    

## Step 6 — Running total per user

Pattern:

```text
PARTITION BY user_id
ORDER BY event_time
ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
```

In [7]:
running_window = (
    Window
    .partitionBy("user_id")
    .orderBy("event_time")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

with_running_total = (
    latest_events
    .withColumn(
        "signed_amount",
        F.when(F.col("event_type") == "refund", -F.col("amount")).otherwise(F.col("amount"))
    )
    .withColumn("running_net_amount", F.sum("signed_amount").over(running_window))
)

with_running_total.orderBy("user_id", "event_time").show(truncate=False)

+--------+-------+----------+------+-------------------+-------------------+-------------+------------------+
|event_id|user_id|event_type|amount|event_time         |ingestion_time     |signed_amount|running_net_amount|
+--------+-------+----------+------+-------------------+-------------------+-------------+------------------+
|e1      |u1     |purchase  |10.0  |2026-01-01 10:00:00|2026-01-01 10:01:00|10.0         |10.0              |
|e2      |u1     |purchase  |17.0  |2026-01-01 10:05:00|2026-01-01 10:07:00|17.0         |27.0              |
|e3      |u1     |refund    |5.0   |2026-01-01 10:10:00|2026-01-01 10:11:00|-5.0         |22.0              |
|e4      |u2     |purchase  |20.0  |2026-01-01 11:00:00|2026-01-01 11:01:00|20.0         |20.0              |
|e5      |u2     |purchase  |30.0  |2026-01-01 11:05:00|2026-01-01 11:06:00|30.0         |50.0              |
|e6      |u2     |refund    |8.0   |2026-01-01 11:10:00|2026-01-01 11:11:00|-8.0         |42.0              |
+--------+

## Step 7 — Control totals

In [8]:
control_totals_schema = StructType([
    StructField("metric", StringType(), False),
    StructField("value", LongType(), False),
])

control_totals = spark.createDataFrame(
    [
        ("input_rows", events.count()),
        ("latest_event_rows", latest_events.count()),
        ("deduped_rows_removed", events.count() - latest_events.count()),
    ],
    control_totals_schema
)

control_totals.show(truncate=False)

+--------------------+-----+
|metric              |value|
+--------------------+-----+
|input_rows          |7    |
|latest_event_rows   |6    |
|deduped_rows_removed|1    |
+--------------------+-----+



## Final result

```text
WINDOW FUNCTIONS
  |
  +--> row_number: latest row / dedup
  +--> dense_rank: ranking inside group
  +--> lag: previous row
  +--> running sum: cumulative metrics
```

In [9]:
spark.stop()